# FutureWork AI — Phase 1: Occupational Skills Data Foundation
**IBM LinuxONE Datathon 2026 · Tech for Good**

**Goal:** Build a clean, reproducible occupation–skill dataset from **O*NET 31.0**. This notebook prepares real data; it does **not** yet claim validated career recommendations.

**Data source:** O*NET 31.0 Database, U.S. Department of Labor/Employment and Training Administration, licensed CC BY 4.0. See https://www.onetcenter.org/database.html and https://creativecommons.org/licenses/by/4.0/ . We transform original occupational ratings into a wide skills matrix and document the changes.

**Important:** O*NET describes U.S. occupations. FutureWork AI must not portray these descriptions as Australia-specific employment or qualification advice.

## 0. Before you run
Download **three CSV files from the same O*NET release (31.0)** using the official database page:

1. `occupation_data.csv` (Occupation Data)
2. `transferable_skills.csv` (Transferable Skills)
3. `essential_skills.csv` (Essential Skills)

Visit **https://www.onetcenter.org/database.html**, choose **CSV** and download the individual files. If the filenames differ slightly, rename them to the names above.

In JupyterLab, create a folder called `data` beside this notebook and **upload all three CSV files into `data`** using the upload button or drag-and-drop.

*Alternatively*, the notebook will look in the current folder or a `data` subfolder. No additional packages should be required beyond `pandas`, `numpy`, and `matplotlib`.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

BASE = Path.cwd()
DATA = BASE / "data"
OUT = BASE / "processed"
OUT.mkdir(exist_ok=True)
print("Notebook working directory:", BASE)
print("Data folder:", DATA)
print("Pandas:", pd.__version__)

In [ ]:
# Find the three uploaded official O*NET CSV files.
def locate(filename):
    candidates = [DATA / filename, BASE / filename]
    for candidate in candidates:
        if candidate.is_file():
            return candidate
    raise FileNotFoundError(
        f"Missing {filename}. Download O*NET 31.0 CSV from "
        "https://www.onetcenter.org/database.html and upload it "
        "into the 'data' folder next to this notebook."
    )

filepaths = {stem: locate(stem + '.csv') for stem in
             ['occupation_data', 'transferable_skills', 'essential_skills']}
for key, path in filepaths.items():
    print(f"Found {key}: {path} ({path.stat().st_size:,} bytes)")

## 1. Load and inspect the data
We use occupation codes (not job titles) as identifiers. Titles can change or duplicate.

In [ ]:
occupations = pd.read_csv(filepaths['occupation_data'], dtype={'O*NET-SOC Code': str}, encoding='utf-8-sig')
transferable = pd.read_csv(filepaths['transferable_skills'], dtype={'O*NET-SOC Code': str}, encoding='utf-8-sig')
essential = pd.read_csv(filepaths['essential_skills'], dtype={'O*NET-SOC Code': str}, encoding='utf-8-sig')
for label, frame in [('Occupations', occupations), ('Transferable skills', transferable), ('Essential skills', essential)]:
    print(f"{label}: {len(frame):,} rows, {len(frame.columns)} columns")
    print('  First columns:', list(frame.columns[:8]))
display(occupations.head(3))

## 2. Standardise and filter skill records
O*NET provides ratings on different **scales**. We use only **Importance** (`IM`) so values are comparable. The original importance ratings generally use a 1–5 scale. We exclude rows flagged as not relevant or recommended for suppression, rather than pretending they are zero.

Essential and transferable skills cover different element groups, so we identify every skill using `Element ID` **and** a namespace.

In [ ]:
expected_occ = {'O*NET-SOC Code', 'Title'}
expected_skills = {'O*NET-SOC Code', 'Element ID', 'Element Name', 'Scale ID', 'Data Value'}
assert expected_occ.issubset(occupations.columns), 'Occupation CSV columns changed: inspect headers.'
assert expected_skills.issubset(transferable.columns), 'Transferable CSV schema unexpected.'
assert expected_skills.issubset(essential.columns), 'Essential CSV schema unexpected.'


def tidy_skill_table(frame, family):
    x = frame.copy()
    x['Scale ID'] = x['Scale ID'].astype(str).str.strip().str.upper()
    x = x.loc[x['Scale ID'].eq('IM')].copy()
    for flag in ['Recommend Suppress', 'Not Relevant']:
        if flag in x.columns:
            x = x.loc[~x[flag].fillna('N').astype(str).str.upper().eq('Y')].copy()
    x['Data Value'] = pd.to_numeric(x['Data Value'], errors='coerce')
    x = x.dropna(subset=['O*NET-SOC Code', 'Element ID', 'Data Value'])
    x = x.loc[x['Data Value'].between(1, 5)].copy()
    x['skill_id'] = family + ':' + x['Element ID'].astype(str)
    x['skill_name'] = x['Element Name'].astype(str)
    x['skill_family'] = family
    return x[['O*NET-SOC Code', 'skill_id', 'skill_name', 'skill_family', 'Data Value']]

skills_long = pd.concat([
    tidy_skill_table(transferable, 'transferable'),
    tidy_skill_table(essential, 'essential'),
], ignore_index=True)
assert not skills_long.empty, 'No Importance ratings found. Check source version and schema.'
print('Usable skill ratings:', len(skills_long))
print('Occupations with ratings:', skills_long['O*NET-SOC Code'].nunique())
print('Distinct skills:', skills_long['skill_id'].nunique())
display(skills_long.head())

## 3. Create an occupation–skill matrix
Rows are occupations, columns are skill descriptors; values are skill **importance ratings**, not evidence of a particular person's proficiency. **Missing values remain missing** in the primary matrix. In Phase 2 we will decide explicitly how to handle missingness for similarity calculations.

In [ ]:
# If an unexpected duplicate exists, averaging avoids reshape failure and flags the issue.
duplicates = skills_long.duplicated(['O*NET-SOC Code', 'skill_id']).sum()
print('Duplicate occupation-skill ratings:', duplicates)
skill_matrix = skills_long.pivot_table(index='O*NET-SOC Code', columns='skill_id', values='Data Value', aggfunc='mean')
skill_matrix.index.name = 'occupation_code'
skill_matrix.columns.name = None

occupation_lookup = occupations[['O*NET-SOC Code', 'Title']].drop_duplicates('O*NET-SOC Code').rename(
    columns={'O*NET-SOC Code': 'occupation_code', 'Title': 'occupation_title'}
)
occupation_lookup = occupation_lookup[occupation_lookup['occupation_code'].isin(skill_matrix.index)].copy()
print('Matrix size:', skill_matrix.shape)
print('Nonmissing proportion:', f'{skill_matrix.notna().mean().mean():.1%}')
display(skill_matrix.iloc[:5, :8])

## 4. Sanity checks and explore coverage

In [ ]:
assert skill_matrix.index.is_unique, 'Occupation codes must be unique.'
assert skill_matrix.columns.is_unique, 'Skill IDs must be unique.'
assert skill_matrix.notna().sum().sum() > 0
assert skill_matrix.min().min() >= 1 and skill_matrix.max().max() <= 5
assert occupation_lookup['occupation_code'].is_unique

coverage = skill_matrix.notna().sum(axis=1)
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(coverage, bins=15)
ax.set(title='Number of rated skills per occupation', xlabel='Available importance ratings', ylabel='Number of occupations')
plt.tight_layout()
plt.show()
print('Minimum, median, maximum rated skills:', int(coverage.min()), float(coverage.median()), int(coverage.max()))

# Preview occupations and skill ratings by choosing a title substring.
search_text = 'sales'
display(occupation_lookup[occupation_lookup['occupation_title'].str.contains(search_text, case=False, na=False)].head(15))

## 5. Save reusable Phase 1 artifacts

In [ ]:
skill_matrix.reset_index().to_csv(OUT / 'occupation_skill_matrix.csv', index=False)
occupation_lookup.to_csv(OUT / 'occupation_lookup.csv', index=False)
skills_long.to_csv(OUT / 'skills_long_clean.csv', index=False)

skill_dictionary = skills_long[['skill_id', 'skill_name', 'skill_family']].drop_duplicates().sort_values('skill_id')
skill_dictionary.to_csv(OUT / 'skill_dictionary.csv', index=False)

print('Phase 1 complete. Files saved in:', OUT)
for path in sorted(OUT.glob('*.csv')):
    print(f' • {path.name} — {path.stat().st_size:,} bytes')

## 6. Notes for Phase 2
- Use **cosine similarity** over appropriate, explicitly imputed or complete skill ratings.
- **Importance ≠ proficiency:** obtain user self-ratings separately; don't interpret occupational importance as evidence of a user's competence.
- Test several occupations and guard against misleadingly high similarity due to commonly important skills.
- Add skill-gap explanations with clear caveats, and filter jobs that require credentials or licences.
- Credit **O*NET 31.0 Database, U.S. Department of Labor/Employment and Training Administration; CC BY 4.0**, and note modifications.
- Australian job market context is a separate future extension, not included in Phase 1.

**Completion check:** Share a screenshot of the final `Phase 1 complete` output and the printed matrix shape. We can then start the recommender.